# S10 — Leonardo | Predicción del tráfico aéreo observado
**Proyecto Sello, Unidad II · CRISP-DM y streaming.** Fuente: observaciones reales OpenSky validadas en S8.

**Pregunta:** ¿Cuántas aeronaves ICAO24 distintas serán observadas en el siguiente intervalo de 5 minutos?

**Advertencia:** se predicen aeronaves *observadas por la API* y no vuelos oficiales ni el tráfico total. Un intervalo sin mensajes puede ser pérdida de muestreo; no se considera cero real por defecto.

**Referencia metodológica:** S10 del docente: `lead()`, división cronológica, persistencia, LinearRegression, RandomForest, modelo guardado y Kafka con `foreachBatch`. Para tráfico agregado, la inferencia necesita *estado de ventanas*; no equivale a la demostración sin estado del docente.

## Fase 1. Business Understanding

In [1]:
NOMBRE_APP = 'S10-Leonardo-Trafico'
from pathlib import Path
import json, math
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, DoubleType)
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ROOT = '/workspace'
RUTA = f'{ROOT}/artifacts/s8/parquet_aeronaves_final'
PAQUETE = 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0'
spark = (SparkSession.builder.appName(NOMBRE_APP).master('local[2]')
    .config('spark.jars.packages', PAQUETE)
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.shuffle.partitions', '3').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version, '| Data Lake:', RUTA)


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ff79197a-cee7-44dc-952b-2490059f4ba7;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark: 4.2.0 | Data Lake: /workspace/artifacts/s8/parquet_aeronaves_final


## Fase 2. Data Understanding — inspeccionar cobertura antes del ML

In [2]:
df = spark.read.parquet(RUTA)
df.printSchema()
print('Observaciones:',df.count(),'Aeronaves:',df.select('icao24').distinct().count())
df.select(F.min('eventTimestamp').alias('inicio'),F.max('eventTimestamp').alias('fin')).show(truncate=False)
df.groupBy(F.to_date('eventTimestamp').alias('fecha')).count().orderBy('fecha').show(30, False)


root
 |-- eventId: string (nullable = true)
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- timePosition: long (nullable = true)
 |-- eventTimestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- baroAltitude: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- verticalRate: double (nullable = true)
 |-- source: string (nullable = true)
 |-- kafkaPartition: integer (nullable = true)
 |-- kafkaOffset: long (nullable = true)



Observaciones: 248 Aeronaves: 49


+-------------------+-------------------+
|inicio             |fin                |
+-------------------+-------------------+
|2026-10-30 15:16:47|2026-10-30 19:58:07|
+-------------------+-------------------+



[Stage 13:======================================>                   (2 + 1) / 3]

+----------+-----+
|fecha     |count|
+----------+-----+
|2026-10-30|248  |
+----------+-----+



## Fase 3. Data Preparation — variables temporales y `label = lead(aeronaves, 1)`
Se usan solo buckets con al menos una observación; se comprueba que el siguiente bucket sea exactamente 5 minutos posterior. No se completan huecos con ceros no verificados. Para una evaluación seria necesitamos muchos intervalos consecutivos.

In [3]:
validos = (df.filter(F.col('eventTimestamp').isNotNull() & F.col('icao24').isNotNull())
    .filter(F.col('latitude').between(-90,90) & F.col('longitude').between(-180,180)))
buckets = (validos.groupBy(F.window('eventTimestamp','5 minutes').alias('w'))
    .agg(F.countDistinct('icao24').alias('aeronaves'),F.avg('velocity').alias('vel_media'),
         F.avg('baroAltitude').alias('alt_media'),F.count('*').alias('observaciones'))
    .select(F.col('w.start').alias('inicio'),F.col('w.end').alias('fin'),
            'aeronaves','vel_media','alt_media','observaciones')
    .withColumn('hora', F.hour('inicio').cast('double'))
    .withColumn('dia_semana', F.dayofweek('inicio').cast('double'))
    .withColumn('hora_sin', F.sin(F.col('hora') * F.lit(2*math.pi/24)))
    .withColumn('hora_cos', F.cos(F.col('hora') * F.lit(2*math.pi/24)))
    .orderBy('inicio'))
buckets.orderBy('inicio').show(20, False)
print('Buckets con datos:', buckets.count())


+-------------------+-------------------+---------+------------------+------------------+-------------+----+----------+-------------------+-----------------------+
|inicio             |fin                |aeronaves|vel_media         |alt_media         |observaciones|hora|dia_semana|hora_sin           |hora_cos               |
+-------------------+-------------------+---------+------------------+------------------+-------------+----+----------+-------------------+-----------------------+
|2026-10-30 15:15:00|2026-10-30 15:20:00|9        |174.47625         |5422.10625        |16           |15.0|6.0       |-0.7071067811865471|-0.7071067811865479    |
|2026-10-30 15:20:00|2026-10-30 15:25:00|14       |176.515           |5575.299999999999 |24           |15.0|6.0       |-0.7071067811865471|-0.7071067811865479    |
|2026-10-30 16:25:00|2026-10-30 16:30:00|9        |156.21625         |3781.4249999999997|16           |16.0|6.0       |-0.8660254037844384|-0.5000000000000004    |
|2026-10-30 16:3

[Stage 22:======================================>                   (2 + 1) / 3]

Buckets con datos: 13


In [4]:
w = Window.orderBy('inicio')
dataset = (buckets
    .withColumn('siguiente_inicio', F.lead('inicio', 1).over(w))
    .withColumn('label', F.lead('aeronaves', 1).over(w).cast('double'))
    .filter(F.col('siguiente_inicio') == F.expr('inicio + INTERVAL 5 MINUTES'))
    .withColumn('persistencia', F.col('aeronaves').cast('double'))
    .fillna({'vel_media':0.0,'alt_media':0.0})
    .orderBy('inicio'))
FEATURES = ['aeronaves','observaciones','vel_media','alt_media','hora_sin','hora_cos','dia_semana']
filas = dataset.count()
print('Pares consecutivos válidos:',filas)
dataset.select('inicio','aeronaves','siguiente_inicio','label').show(20,False)
if filas < 30:
    raise RuntimeError(f'Solo {filas} pares válidos; recopila más días/intervalos antes de entrenar. No rellenes vacíos con ceros inventados.')


Pares consecutivos válidos: 9


+-------------------+---------+-------------------+-----+
|inicio             |aeronaves|siguiente_inicio   |label|
+-------------------+---------+-------------------+-----+
|2026-10-30 15:15:00|9        |2026-10-30 15:20:00|14.0 |
|2026-10-30 16:25:00|9        |2026-10-30 16:30:00|12.0 |
|2026-10-30 16:30:00|12       |2026-10-30 16:35:00|14.0 |
|2026-10-30 18:30:00|4        |2026-10-30 18:35:00|6.0  |
|2026-10-30 19:30:00|1        |2026-10-30 19:35:00|9.0  |
|2026-10-30 19:35:00|9        |2026-10-30 19:40:00|7.0  |
|2026-10-30 19:40:00|7        |2026-10-30 19:45:00|10.0 |
|2026-10-30 19:45:00|10       |2026-10-30 19:50:00|12.0 |
|2026-10-30 19:50:00|12       |2026-10-30 19:55:00|9.0  |
+-------------------+---------+-------------------+-----+



RuntimeError: Solo 9 pares válidos; recopila más días/intervalos antes de entrenar. No rellenes vacíos con ceros inventados.

### División temporal 80/20, sin muestreo aleatorio

In [5]:
orden = Window.orderBy('inicio')
partes = dataset.withColumn('_orden',F.row_number().over(orden))
limite = int(filas * 0.8)
train = partes.filter(F.col('_orden')<=limite).drop('_orden').cache()
test  = partes.filter(F.col('_orden')>limite).drop('_orden').cache()
print('Train:',train.count(),'Test:',test.count())
train.agg(F.max('inicio')).show(); test.agg(F.min('inicio')).show()


Train: 7 Test: 2
+-------------------+
|        max(inicio)|
+-------------------+
|2026-10-30 19:40:00|
+-------------------+

+-------------------+
|        min(inicio)|
+-------------------+
|2026-10-30 19:45:00|
+-------------------+



## Fases 4 y 5. Baseline, LinearRegression y RandomForest

In [6]:
# 4. Modelado y 5. Evaluación. El test es estrictamente posterior al train.
def evaluar(df, columna='prediction'):
    return {k: RegressionEvaluator(labelCol='label', predictionCol=columna, metricName=k).evaluate(df)
            for k in ('rmse', 'mae', 'r2')}

base = evaluar(test.withColumn('prediction', F.col('persistencia')))
print('Baseline persistencia:', base)
modelos = {}
for nombre, estimador in [
    ('LinearRegression', LinearRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.01)),
    ('RandomForest', RandomForestRegressor(featuresCol='features', labelCol='label', numTrees=40, maxDepth=5, seed=42))
]:
    pipeline = Pipeline(stages=[VectorAssembler(inputCols=FEATURES, outputCol='features', handleInvalid='skip'), estimador])
    modelo = pipeline.fit(train)
    met = evaluar(modelo.transform(test))
    modelos[nombre] = (modelo, met)
    print(nombre, met)

ordenados = sorted(modelos, key=lambda k: modelos[k][1]['rmse'])
mejor_ml = ordenados[0]
seleccion = 'Persistencia' if base['rmse'] <= modelos[mejor_ml][1]['rmse'] else mejor_ml
print('Mejor candidato segun RMSE:', seleccion)
print('ATENCIÓN: evaluar R2, MAE, estabilidad y tamaño del test antes de confiar en el modelo.')


Baseline persistencia: {'rmse': 2.5495097567963922, 'mae': 2.5, 'r2': -1.8888888888888884}


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


LinearRegression {'rmse': 3.6752352926254153, 'mae': 3.6369819447999054, 'r2': -5.003268647181965}
RandomForest {'rmse': 2.230626469054029, 'mae': 1.875, 'r2': -1.2114197530864201}
Mejor candidato segun RMSE: RandomForest
ATENCIÓN: evaluar R2, MAE, estabilidad y tamaño del test antes de confiar en el modelo.


## Fase 6. Persistencia del mejor modelo ML
Incluso si gana la persistencia, se guarda el mejor *PipelineModel de MLlib* para demostrar el ciclo de despliegue. El sistema operativo usará el candidato elegido según RMSE.

In [7]:
MODEL_DIR = f'{ROOT}/artifacts/s10/models/leonardo_trafico_ml'
modelos[mejor_ml][0].write().overwrite().save(MODEL_DIR)
recargado = PipelineModel.load(MODEL_DIR)
recargado.transform(test).select('inicio','label','prediction').show(10, False)
Path(f'{ROOT}/artifacts/s10').mkdir(parents=True,exist_ok=True)
Path(f'{ROOT}/artifacts/s10/seleccion_leonardo.json').write_text(json.dumps({
    'elegido':seleccion,'mejor_ml':mejor_ml,'baseline':base,
    'modelos':{k:v[1] for k,v in modelos.items()},'pares_validos':filas
},indent=2))
print('Modelo guardado y recargado:',MODEL_DIR)


+-------------------+-----+-----------------+
|inicio             |label|prediction       |
+-------------------+-----+-----------------+
|2026-10-30 19:45:00|12.0 |8.916666666666666|
|2026-10-30 19:50:00|9.0  |9.666666666666666|
+-------------------+-----+-----------------+

Modelo guardado y recargado: /workspace/artifacts/s10/models/leonardo_trafico_ml


## Fase 7. Inferencia en streaming desde Kafka
**Ejecución opcional y continua**: crea predicciones solo para ventanas cerradas por watermark. Se necesitan eventos posteriores para avanzar el watermark. No iniciar otra copia de este mismo stream/checkpoint. Se usa `foreachBatch` para evaluar el modelo en cada lote y guardar resultados de prueba. El valor de predicción corresponde al siguiente bucket tras la ventana finalizada.

In [8]:
ESQUEMA = StructType([
    StructField('eventId',StringType()), StructField('eventType',StringType()),
    StructField('schemaVersion',IntegerType()), StructField('icao24',StringType()),
    StructField('eventTime',StringType()), StructField('latitude',DoubleType()),
    StructField('longitude',DoubleType()), StructField('velocity',DoubleType()),
    StructField('baroAltitude',DoubleType()), StructField('source',StringType())])

raw = (spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers','kafka:9092')
    .option('subscribe','aeronaves-telemetria')
    .option('startingOffsets','latest').load())
events = (raw.select(F.from_json(F.col('value').cast('string'),ESQUEMA).alias('d'))
    .select('d.*')
    .withColumn('eventTimestamp',F.to_timestamp('eventTime'))
    .filter((F.col('eventType')=='aeronave.telemetria') & (F.col('schemaVersion')==1) & (F.col('source')=='opensky'))
    .filter(F.col('eventTimestamp').isNotNull() & F.col('icao24').rlike('^[0-9a-f]{6}$'))
    .filter(F.col('latitude').between(-90,90) & F.col('longitude').between(-180,180))
    .withWatermark('eventTimestamp','2 minutes'))

# Spark en streaming no soporta countDistinct exacto como agregacion en general.
ventanas = (events.groupBy(F.window('eventTimestamp','5 minutes'))
    .agg(F.approx_count_distinct('icao24').alias('aeronaves'),
         F.count('*').alias('observaciones'),F.avg('velocity').alias('vel_media'),
         F.avg('baroAltitude').alias('alt_media'))
    .select(F.col('window.start').alias('inicio'), F.col('window.end').alias('fin'),
            'aeronaves','observaciones','vel_media','alt_media')
    .withColumn('hora',F.hour('inicio').cast('double'))
    .withColumn('dia_semana',F.dayofweek('inicio').cast('double'))
    .withColumn('hora_sin',F.sin(F.col('hora')*F.lit(2*math.pi/24)))
    .withColumn('hora_cos',F.cos(F.col('hora')*F.lit(2*math.pi/24)))
    .fillna({'vel_media':0.0,'alt_media':0.0}))

def predecir_lote(lote,batch_id):
    if lote.isEmpty(): return
    pred = (lote.withColumn('prediction',F.col('aeronaves').cast('double'))
            if seleccion=='Persistencia' else recargado.transform(lote))
    resultado = (pred.select('inicio','fin','aeronaves','prediction')
        .withColumn('objetivo_inicio',F.col('fin'))
        .withColumn('modelo',F.lit(seleccion))
        .withColumn('batch_id',F.lit(batch_id)))
    resultado.show(truncate=False)
    resultado.write.mode('append').parquet(f'{ROOT}/artifacts/s10/predicciones/leonardo')

print('Para empezar la escucha, ejecuta la celda siguiente; detener con stream.stop().')


Para empezar la escucha, ejecuta la celda siguiente; detener con stream.stop().


In [9]:
# EJECUTAR SOLO CUANDO YA HAYAS COMPLETADO EL ENTRENAMIENTO.
# Esta celda regresa inmediatamente. Verifica stream.isActive.
stream = (ventanas.writeStream.outputMode('append').foreachBatch(predecir_lote)
    .option('checkpointLocation',f'{ROOT}/artifacts/checkpoints/s10_leonardo_v1')
    .trigger(processingTime='30 seconds').start())
print('Stream iniciado:', stream.isActive)
# Para detener: stream.stop()


Stream iniciado: True


+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 20:00:00|2026-10-30 20:05:00|4        |9.975     |2026-10-30 20:05:00|RandomForest|6       |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 20:05:00|2026-10-30 20:10:00|3        |9.9       |2026-10-30 20:10:00|RandomForest|12      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 20:10:00|2026-10-30 20:15:00|1        |10.1      |2026-10-30 20:15:00|RandomForest|14      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 20:15:00|2026-10-30 20:20:00|2        |10.1      |2026-10-30 20:20:00|RandomForest|18      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 20:20:00|2026-10-30 20:25:00|3        |10.05     |2026-10-30 20:25:00|RandomForest|24      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



... (12 outputs left out)

+-------------------+-------------------+---------+------------------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction        |objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+------------------+-------------------+------------+--------+
|2026-10-30 21:25:00|2026-10-30 21:30:00|10       |11.041666666666668|2026-10-30 21:30:00|RandomForest|88      |
+-------------------+-------------------+---------+------------------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 21:30:00|2026-10-30 21:35:00|6        |10.55     |2026-10-30 21:35:00|RandomForest|94      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 21:35:00|2026-10-30 21:40:00|5        |10.275    |2026-10-30 21:40:00|RandomForest|98      |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|inicio             |fin                |aeronaves|prediction|objetivo_inicio    |modelo      |batch_id|
+-------------------+-------------------+---------+----------+-------------------+------------+--------+
|2026-10-30 21:40:00|2026-10-30 21:45:00|8        |10.85     |2026-10-30 21:45:00|RandomForest|104     |
+-------------------+-------------------+---------+----------+-------------------+------------+--------+



## Interpretación y limitaciones
Anota aquí MAE/RMSE/R² de los tres candidatos, qué modelo ganó y por qué. Señala huecos temporales, número de pares y sesgo de cobertura OpenSky. No anuncies precisión predictiva si no hay suficientes pares de intervalos.